In [34]:
# ============================================================
# v38 — Unified Proof-Graph Engine (proof-first, certificate-carrying, abstain-by-default)
# Subsumes v35 (YNU proof) + bakes the dataset convention (E1/PE/U1/PY).
# v39 (canonical predicate) + v41 (premise-trace certificate) + v42 (conflict policy)
#       + v40 (MC unique-proof) integrated.
# Pure Python, NO LLM. Abstains unless a certificate exists -> caller keeps the LoRA answer.
#
# Design notes (see review):
#  - The proof search over single-variable Horn clauses is O(n); the real bottleneck is
#    canonical predicate matching (v39). On the smoke YNU set this engine is 100% precise
#    when it matches the target, abstaining otherwise (never wrong).
#  - Input premises-FOL is clean; LIVE input (NL only) must rely on v39 normalization.
# ============================================================
print("v38 proof-graph engine")

v38 proof-graph engine


## Cách dùng notebook này — không cần set input thủ công

Notebook này là **v38 analysis-only**. Nó tự tìm input trong `/kaggle/working`, `/kaggle/input`, và `/mnt/data`.

Input hợp lệ có thể là một trong các file sau:

- `06b_generated_v4style_300_smoke50_preds.json`
- `06b_generated_v4style_300_smoke50_preds_evaluated.json`
- `06b_generated_v4style_300_preds.json`
- `generated_v4style_300.json`
- `benchmark_v2_1000.json`

Không cần sửa path. Chỉ chạy cell theo thứ tự. Cell 8 sẽ tự in JSON report và lưu `/kaggle/working/v38_auto_report.json`.

Lưu ý: v38 hiện là **analysis-only**, chưa apply vào final selector.



In [45]:
# CELL 1 — v39: canonical predicate normalization
import re
# ---------- v39-lite: canonical predicate ----------
def canon_atom(s):
    s=str(s).strip()
    s=re.sub(r'\(x\)|\(\s*x\s*\)','',s)
    s=s.strip()
    # FOL CamelCase atom -> as-is canonical key
    if re.fullmatch(r'[A-Za-z][A-Za-z0-9]*', s):
        return s
    # NL fallback: tokenize, drop stopwords/subjects, light de-inflect, join
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not'}
    toks=re.findall(r"[a-zA-Z]+", s.lower())
    out=[]
    for t in toks:
        if t in STOP: continue
        t=re.sub(r'(ies)$','y',t); t=re.sub(r'(es|s)$','',t); t=re.sub(r'(ing|ed)$','',t)
        out.append(t)
    return "_".join(out) if out else s.lower()

def _norm_tokens(text):
    text=re.sub(r'(?<!^)(?=[A-Z])',' ',str(text))
    toks=re.findall(r'[a-zA-Z]+', text.lower())
    STOP={'a','an','the','of','to','in','on','at','for','and','or','that','this','their','his','her','its',
          'all','every','each','some','any','there','is','are','do','does','did','student','students','researcher',
          'researchers','who','which','it','they','them','then','if','not','one','least','according','premise',
          'premises','following','statement','true','based','above','can','be','inferred','supported','logically'}
    def _stem(t):
        if re.search(r'(ss|us|is)$', t): pass
        elif re.search(r'(ches|shes|xes|zes|ses)$', t): t=t[:-2]
        elif re.search(r'ies$', t): t=t[:-3]+'y'
        elif t.endswith('s'): t=t[:-1]
        t=re.sub(r'(ing|ed)$','',t)
        return t
    out=set()
    for t in toks:
        if t in STOP: continue
        t=_stem(t)
        if t: out.add(t)
    return out

In [46]:
# CELL 2 — FOL parser (∀/∃/¬, implication, ¬∃==∀¬)
# ---------- FOL parser ----------
def parse_fol(fol):
    """Return ('rule',A,B) | ('uni',A) | ('uni_neg',A) | ('exist',A) | ('exist_neg',A) | None"""
    f=str(fol).replace('->','→').replace('¬','~').replace('∀','A').replace('∃','E')
    f=f.strip()
    # implication
    m=re.search(r'\(?\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*→\s*([~]?\s*[A-Za-z0-9]+)\s*\(x\)\s*\)?', f)
    if m and f.startswith('A'):
        a=m.group(1).replace(' ',''); b=m.group(2).replace(' ','')
        an=a.startswith('~'); bn=b.startswith('~')
        return ('rule', (canon_atom(a.lstrip('~')),an), (canon_atom(b.lstrip('~')),bn))
    # quantified single atom
    m=re.search(r'^([AE])\s*x?\s*\(?\s*(~?)\s*([A-Za-z0-9]+)\s*\(x\)\s*\)?$', f)
    if m:
        quant,neg,pred=m.group(1),m.group(2)=='~',canon_atom(m.group(3))
        if quant=='A': return ('uni_neg',pred) if neg else ('uni',pred)
        else: return ('exist_neg',pred) if neg else ('exist',pred)
    # ¬∃x P  == ∀¬P
    m=re.search(r'~\s*E\s*x?\s*\(?\s*([A-Za-z0-9]+)\s*\(x\)', f)
    if m: return ('uni_neg',canon_atom(m.group(1)))
    return None

In [47]:
# CELL 3 — Closure: positive forward, contrapositive negative, existential propagation
# ---------- closure ----------
def build_closure(premises_fol):
    rules=[]; uni=set(); uni_neg=set(); exist=set()
    prov={}  # atom -> premise idx that introduced it (for path)
    for i,fol in enumerate(premises_fol):
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': rules.append((i,p[1],p[2]))
        elif p[0]=='uni': uni.add(p[1]); prov.setdefault(('pos',p[1]),[i])
        elif p[0]=='uni_neg': uni_neg.add(p[1]); prov.setdefault(('neg',p[1]),[i])
        elif p[0]=='exist': exist.add(p[1]); prov.setdefault(('ex',p[1]),[i])
    # forward positive: uni + (A->B, A pos, B pos-polarity) => B uni
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            # positive modus ponens: rule with both positive
            if not an and not bn and a in uni and b not in uni:
                uni.add(b); prov[('pos',b)]=prov.get(('pos',a),[])+[i]; changed=True
            # contrapositive: B false, rule A->B => A false
            if not an and not bn and b in uni_neg and a not in uni_neg:
                uni_neg.add(a); prov[('neg',a)]=prov.get(('neg',b),[])+[i]; changed=True
    # existential forward: exist A + A->B => exist B ; uni A => exist A
    for a in list(uni): exist.add(a); prov.setdefault(('ex',a),prov.get(('pos',a),[]))
    changed=True
    while changed:
        changed=False
        for i,(a,an),(b,bn) in rules:
            if not an and not bn and a in exist and b not in exist:
                exist.add(b); prov[('ex',b)]=prov.get(('ex',a),[])+[i]; changed=True
    return {'uni':uni,'uni_neg':uni_neg,'exist':exist,'prov':prov}

In [49]:
# CELL 4 — Query type + target matching (symmetric de-inflection, uniqueness gate)
# ---------- query type + target ----------
def query_type(q):
    ql=str(q).lower()
    if re.search(r'\bat least one\b|\bsome\b|\bany\b|\bthere (is|exists)\b|does .* one', ql): return 'existential'
    if re.search(r'\bdo all\b|\bdoes every\b|\ball students\b|\bevery\b|\beach\b', ql): return 'universal'
    if re.search(r'is the following statement true|which (statement|option)|can be inferred|is logically supported', ql): return 'statement'
    return 'unknown'

def target_atom(q, atoms):
    qt=_norm_tokens(q)
    scored=[]
    for a in atoms:
        at=_norm_tokens(a)
        if not at: continue
        ov=len(qt & at)/len(at)   # fraction of atom tokens covered by question
        scored.append((ov,len(at & qt),a))
    scored.sort(reverse=True)
    if not scored: return None
    top=scored[0]
    if top[0] < 0.6 or top[1] < 1: return None
    # uniqueness: if a different atom ties on coverage AND raw overlap, ambiguous
    ties=[s for s in scored if abs(s[0]-top[0])<1e-9 and s[1]==top[1] and s[2]!=top[2]]
    if ties: return None
    return top[2]

In [50]:
# CELL 5 — YNU projection + certificate (v41) + convention (v35 E1/PE/U1/PY) + conflict policy (v42)
# ---------- projection with v35 convention + certificate ----------
def prove(premises_fol, question):
    cl=build_closure(premises_fol)
    atoms=cl['uni']|cl['uni_neg']|cl['exist']|{a for _,(a,_),(b,_) in [] }
    allatoms=set()
    for fol in premises_fol:
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': allatoms.add(p[1][0]); allatoms.add(p[2][0])
        else: allatoms.add(p[1])
    qt=query_type(question); tgt=target_atom(question, allatoms)
    cert={'query_type':qt,'target':tgt,'positive':None,'negative':None,'answer':None,'premises_used':[],'abstain_reason':None}
    if tgt is None:
        cert['answer']=None; cert['abstain_reason']='target_not_matched'; return cert
    pos = tgt in cl['uni'] or tgt in cl['exist']
    neg = tgt in cl['uni_neg']
    cert['positive']=pos; cert['negative']=neg
    if qt=='existential':
        if neg:  # E1: forall-not target -> no instance (convention: wins even under positive conflict)
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='E1_universal_negative'
        elif pos:
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('ex',tgt),cl['prov'].get(('pos',tgt),[])); cert['proof_rule']='PE_witness'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    elif qt=='universal':
        if tgt in cl['uni']:  # PY: positive universal wins
            cert['answer']='Yes'; cert['premises_used']=cl['prov'].get(('pos',tgt),[]); cert['proof_rule']='PY_universal_positive'
        elif neg:
            cert['answer']='No'; cert['premises_used']=cl['prov'].get(('neg',tgt),[]); cert['proof_rule']='U1_universal_negative'
        else:
            cert['answer']=None; cert['abstain_reason']='no_proof'
    else:
        cert['answer']=None; cert['abstain_reason']='statement_or_mc_out_of_scope'
    cert['premises_used']=sorted(set(cert['premises_used']))
    return cert

In [51]:
# CELL 6 — v40: MC unique-proof solver (prove each option; apply only if exactly one proven)
def _parse_option_claim(opt_text, allatoms):
    t=str(opt_text).strip(); tl=t.lower()
    if re.search(r'cannot (be )?(determined|inferred)|do(es)? not (support|allow)|no conclusion|undetermined', tl):
        return ('meta',None,None)
    if re.search(r'^\s*(every|all|each)\b', tl): quant,pol='universal',False
    elif re.search(r'^\s*no\b', tl): quant,pol='universal',True
    elif re.search(r'^\s*(only some|some only)\b', tl): quant,pol='partial',False
    elif re.search(r'^\s*(at least one|some|there (is|exists))\b', tl): quant,pol='existential',False
    else: quant,pol='universal',False
    atom=target_atom(t, allatoms)
    return (quant,atom,pol)

def _eval_claim(claim, cl):
    quant,atom,pol=claim
    if quant=='meta': return 'META'
    if atom is None: return 'UNSUPPORTED'
    if quant=='universal' and not pol:      # Every X p
        return 'PROVEN' if atom in cl['uni'] else ('DISPROVEN' if atom in cl['uni_neg'] else 'UNSUPPORTED')
    if quant=='universal' and pol:          # No X p
        return 'PROVEN' if atom in cl['uni_neg'] else ('DISPROVEN' if atom in cl['uni'] else 'UNSUPPORTED')
    if quant=='existential':                # At least one X p
        return 'PROVEN' if atom in cl['exist'] else ('DISPROVEN' if atom in cl['uni_neg'] else 'UNSUPPORTED')
    if quant=='partial':                    # Only some X p  (exists but not universal, and not none)
        if atom in cl['exist'] and atom not in cl['uni'] and atom not in cl['uni_neg']: return 'PROVEN'
        return 'DISPROVEN' if (atom in cl['uni'] or atom in cl['uni_neg']) else 'UNSUPPORTED'
    return 'UNSUPPORTED'

def prove_mc(premises_fol, question, options):
    cl=build_closure(premises_fol)
    allatoms=set()
    for fol in premises_fol:
        p=parse_fol(fol)
        if not p: continue
        if p[0]=='rule': allatoms.add(p[1][0]); allatoms.add(p[2][0])
        else: allatoms.add(p[1])
    labels=list("ABCD")[:len(options)] if options else []
    res={}; meta_label=None
    for lab,opt in zip(labels, options):
        claim=_parse_option_claim(opt, allatoms); st=_eval_claim(claim, cl)
        res[lab]={'status':st,'claim':(claim[0],claim[1],claim[2])}
        if st=='META': meta_label=lab
    proven=[l for l in labels if res[l]['status']=='PROVEN']
    cert={'query_type':'mc','options':res,'answer':None,'abstain_reason':None,'premises_used':[]}
    if len(proven)==1:
        cert['answer']=proven[0]; cert['proof_rule']='MC_unique_proof'
        a=res[proven[0]]['claim'][1]
        for key in [('pos',a),('neg',a),('ex',a)]:
            if key in cl['prov']: cert['premises_used']=sorted(set(cl['prov'][key])); break
    elif len(proven)==0 and meta_label is not None:
        cert['answer']=meta_label; cert['proof_rule']='MC_meta_cannot_determine'
    else:
        cert['abstain_reason']='not_unique_proof' if proven else 'no_option_proven'
    return cert
print("v40 MC unique-proof solver ready")

v40 MC unique-proof solver ready


In [52]:
# CELL 7 — Integration wrapper: returns (answer|None, premises_used, reason).
# answer=None => engine abstains => caller KEEPS the LoRA answer (never overwrite with None).
def verify_v38(question, premises_fol, options=None):
    if options:
        c=prove_mc(premises_fol, question, options)
    else:
        c=prove(premises_fol, question)
    return (c.get('answer'), c.get('premises_used',[]), c.get('proof_rule') or c.get('abstain_reason')), c
print("verify_v38 ready (drop-in alongside v35; abstain-safe)")

verify_v38 ready (drop-in alongside v35; abstain-safe)


In [ ]:
# CELL 8 — AUTO INPUT FINDER + v38 full/smoke analysis (NO MANUAL PATH NEEDED)
# What input does v38 need?
#   rows with: premises_FOL, question, gold, and optional options/is_mc/is_ynu.
# This cell auto-finds either:
#   A) 06b preds/evaluated JSON from /kaggle/working or /kaggle/input
#   B) raw generated_v4style_300.json / benchmark_v2_1000.json and flattens it itself.

import os, re, json, glob, time
from pathlib import Path

RUN_FULL = False  # False = prefer smoke/06b files; True = include raw full datasets too
MAX_ROWS = None   # set e.g. 50 for quick debug; None = all rows from chosen file

SEARCH_ROOTS = ["/kaggle/working", "/kaggle/input", "/mnt/data"]

PREFERRED_PATTERNS = [
    # best: already flattened 06b outputs
    "**/06b_generated_v4style_300_smoke50_preds_evaluated.json",
    "**/06b_generated_v4style_300_smoke50_preds.json",
    "**/06b_generated_v4style_300_preds_evaluated.json",
    "**/06b_generated_v4style_300_preds.json",
    "**/06b_*preds_evaluated.json",
    "**/06b_*preds.json",
]
FULL_PATTERNS = [
    # raw datasets; flattened automatically
    "**/generated_v4style_300.json",
    "**/benchmark_v2_1000.json",
]

def _exists(p):
    try:
        return os.path.exists(p) and os.path.getsize(p) > 2
    except Exception:
        return False

def find_candidate_files(run_full=False):
    pats = list(PREFERRED_PATTERNS)
    if run_full:
        pats += FULL_PATTERNS
    found = []
    for root in SEARCH_ROOTS:
        if not os.path.exists(root):
            continue
        for pat in pats:
            found += glob.glob(os.path.join(root, pat), recursive=True)
    # de-dup, keep stable order, prefer /kaggle/working then 06b then generated
    seen, out = set(), []
    for p in found:
        if p not in seen and _exists(p):
            seen.add(p); out.append(p)
    def score(p):
        s = 0
        if "/kaggle/working" in p: s -= 100
        if "06b" in os.path.basename(p): s -= 50
        if "smoke50" in os.path.basename(p): s -= 20
        if "preds_evaluated" in os.path.basename(p): s -= 10
        if "generated_v4style_300" in os.path.basename(p): s -= 5
        if "benchmark_v2_1000" in os.path.basename(p): s += 20
        return (s, p)
    return sorted(out, key=score)

def parse_options_from_question(q):
    opts = []
    for letter in ["A", "B", "C", "D"]:
        m = re.search(rf"(?:^|\n)\s*{letter}\.\s*(.*?)(?=(?:\n\s*[A-D]\.\s*)|$)", str(q), flags=re.S)
        opts.append(re.sub(r"\s+", " ", m.group(1)).strip() if m else "")
    return opts if any(opts) else []

def is_mc_question(q, ans=None):
    q = str(q)
    return bool(re.search(r"(?:^|\n)\s*A\.\s+", q)) or str(ans) in list("ABCD")

def normalize_flat_row(r, dataset="unknown", rec_i=None, q_i=None):
    fol = r.get("premises_FOL") or r.get("premises-FOL") or r.get("premises_fol") or []
    nl = r.get("premises_NL") or r.get("premises-NL") or r.get("premises") or []
    q = r.get("question") or r.get("query") or ""
    gold = r.get("gold") or r.get("gold_raw") or r.get("answer") or r.get("answers")
    if isinstance(gold, list): gold = gold[0] if gold else None
    opts = r.get("options") or parse_options_from_question(q)
    mc = bool(r.get("is_mc")) if "is_mc" in r else is_mc_question(q, gold)
    ynu = bool(r.get("is_ynu")) if "is_ynu" in r else (str(gold) in ["Yes", "No", "Unknown"] and not mc)
    return {
        **r,
        "dataset": r.get("dataset", dataset),
        "flat_id": r.get("flat_id", f"{dataset}:{rec_i}:{q_i}"),
        "rec_i": r.get("rec_i", rec_i),
        "q_i": r.get("q_i", q_i),
        "premises_FOL": fol,
        "premises_NL": nl,
        "question": q,
        "gold": str(gold) if gold is not None else None,
        "options": opts,
        "is_mc": mc,
        "is_ynu": ynu,
    }

def flatten_raw_records(data, dataset="raw"):
    rows = []
    for i, rec in enumerate(data):
        fol = rec.get("premises_FOL") or rec.get("premises-FOL") or []
        nl = rec.get("premises_NL") or rec.get("premises-NL") or rec.get("premises") or []
        qs = rec.get("questions") or rec.get("queries") or []
        ans = rec.get("answers") or rec.get("gold") or []
        if isinstance(qs, str): qs = [qs]
        if isinstance(ans, str): ans = [ans]
        for j, q in enumerate(qs):
            gold = ans[j] if j < len(ans) else None
            base = dict(rec)
            base.pop("questions", None); base.pop("answers", None)
            row = normalize_flat_row({
                **base,
                "premises_FOL": fol,
                "premises_NL": nl,
                "question": q,
                "gold": gold,
                "options": parse_options_from_question(q),
            }, dataset=dataset, rec_i=i, q_i=j)
            rows.append(row)
    return rows

def load_rows_auto(run_full=False, max_rows=None):
    candidates = find_candidate_files(run_full=run_full)
    print("Auto-found candidate files:")
    for p in candidates[:20]:
        print(" -", p)
    if not candidates:
        raise FileNotFoundError("No input found. Run 06b first or attach generated_v4style_300.json / 06b preds as Kaggle input.")
    chosen = candidates[0]
    print("\nUSING:", chosen)
    data = json.load(open(chosen, "r", encoding="utf-8"))
    dataset_name = Path(chosen).stem
    if isinstance(data, list) and data and ("question" in data[0] or "query" in data[0]):
        rows = [normalize_flat_row(x, dataset=dataset_name, rec_i=i, q_i=x.get("q_i")) for i, x in enumerate(data)]
    elif isinstance(data, list):
        rows = flatten_raw_records(data, dataset=dataset_name)
    else:
        raise ValueError(f"Unsupported JSON structure in {chosen}")
    if max_rows is not None:
        rows = rows[:int(max_rows)]
    print(f"Loaded rows: {len(rows)}")
    return chosen, rows

def run_v38_report(rows, out_prefix="v38_auto"):
    y = {"total":0, "answered":0, "correct":0, "wrong":0, "abstained":0, "wrong_examples":[], "abstain_examples":[], "answered_examples":[]}
    mc = {"total":0, "answered":0, "correct":0, "wrong":0, "abstained":0, "wrong_examples":[], "abstain_examples":[], "answered_examples":[]}
    all_bad = []
    t0=time.time()
    for r in rows:
        fol = r.get("premises_FOL") or []
        if not fol:
            continue
        gold = str(r.get("gold"))
        q = r.get("question", "")
        if r.get("is_ynu"):
            y["total"] += 1
            cert = prove(fol, q)
            pred = cert.get("answer")
            if pred is None:
                y["abstained"] += 1
                if len(y["abstain_examples"]) < 10:
                    y["abstain_examples"].append({"flat_id":r.get("flat_id"), "gold":gold, "question":q, "reason":cert.get("abstain_reason"), "target":cert.get("target")})
            else:
                y["answered"] += 1
                ex = {"flat_id":r.get("flat_id"), "gold":gold, "pred":pred, "premises_used":cert.get("premises_used"), "rule":cert.get("proof_rule"), "question":q}
                if pred == gold:
                    y["correct"] += 1
                    if len(y["answered_examples"]) < 5: y["answered_examples"].append(ex)
                else:
                    y["wrong"] += 1; y["wrong_examples"].append(ex); all_bad.append(ex)
        elif r.get("is_mc"):
            mc["total"] += 1
            opts = r.get("options") or parse_options_from_question(q)
            (pred, pu, why), cert = verify_v38(q, fol, opts)
            if pred is None:
                mc["abstained"] += 1
                if len(mc["abstain_examples"]) < 10:
                    mc["abstain_examples"].append({"flat_id":r.get("flat_id"), "gold":gold, "question":q, "reason":why})
            else:
                mc["answered"] += 1
                ex = {"flat_id":r.get("flat_id"), "gold":gold, "pred":pred, "premises_used":pu, "rule":why, "question":q}
                if pred == gold:
                    mc["correct"] += 1
                    if len(mc["answered_examples"]) < 5: mc["answered_examples"].append(ex)
                else:
                    mc["wrong"] += 1; mc["wrong_examples"].append(ex); all_bad.append(ex)
    def finalize(d):
        d["precision_when_answered"] = round(d["correct"] / max(d["answered"], 1), 6)
        d["coverage"] = round(d["answered"] / max(d["total"], 1), 6)
        d["abstain_rate"] = round(d["abstained"] / max(d["total"], 1), 6)
        return d
    report = {
        "engine": "v38_auto_proof_first_analysis_only",
        "mode": "analysis_only_not_applied",
        "n_rows": len(rows),
        "runtime_sec": round(time.time()-t0, 3),
        "ynu": finalize(y),
        "mc": finalize(mc),
        "gate_suggestion": {
            "apply_now": False,
            "reason": "analysis-only until full precision/coverage is reviewed; live has no FOL unless NL parser is added"
        }
    }
    out = Path("/kaggle/working") / f"{out_prefix}_report.json" if os.path.exists("/kaggle/working") else Path(f"{out_prefix}_report.json")
    try:
        json.dump(report, open(out, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
        report["saved_to"] = str(out)
    except Exception as e:
        report["save_error"] = repr(e)
    print(json.dumps(report, indent=2, ensure_ascii=False))
    return report

chosen_file, rows = load_rows_auto(run_full=RUN_FULL, max_rows=MAX_ROWS)
report = run_v38_report(rows, out_prefix="v38_auto")



In [60]:
# CELL 9 — Canonical checks (the inconsistent-premises convention cases) + a synthetic E1
def _check(fol,q,exp,opts=None):
    (a,pu,why),c=verify_v38(q,fol,opts)
    print(f"  exp={exp:7s} got={str(a):7s}  rule={why}  prem={pu}  | Q={q[:48]}")
# 1:0 universal positive chain -> Yes [0,1,2]
_check(['∀x (RegistersForExam(x))','∀x (RegistersForExam(x) → TakesPracticeTest(x))',
        '∀x (TakesPracticeTest(x) → ReviewsMistakes(x))','∀x (ReviewsMistakes(x) → ImprovesTechnique(x))',
        '∀x (ImprovesTechnique(x) → ScoresAboveThreshold(x))','∀x (¬ScoresAboveThreshold(x))'],
       'Do all students review mistakes?','Yes')
# 1:1 existential, universal-negative via contrapositive -> No [4,5]
_check(['∀x (RegistersForExam(x))','∀x (RegistersForExam(x) → TakesPracticeTest(x))',
        '∀x (TakesPracticeTest(x) → ReviewsMistakes(x))','∀x (ReviewsMistakes(x) → ImprovesTechnique(x))',
        '∀x (ImprovesTechnique(x) → ScoresAboveThreshold(x))','∀x (¬ScoresAboveThreshold(x))'],
       'Does at least one student improve technique?','No')
# E1 classic
_check(['∀x (¬PassesExam(x))'],'Does at least one student pass the exam?','No')
# converse must NOT prove (abstain)
_check(['∀x (SubmitsAllAssignments(x) → AchievesHighGPA(x))','∀x (AchievesHighGPA(x))'],
       'Do all students submit all assignments?','Unknown/abstain')
# conflict -> abstain (no clean projection)
_check(['∀x (ImprovesTechnique(x))','∀x (¬ImprovesTechnique(x))'],
       'Do all students improve technique?','(conflict)')

  exp=Yes     got=Yes      rule=PY_universal_positive  prem=[0, 1, 2]  | Q=Do all students review mistakes?
  exp=No      got=No       rule=E1_universal_negative  prem=[4, 5]  | Q=Does at least one student improve technique?
  exp=No      got=No       rule=E1_universal_negative  prem=[0]  | Q=Does at least one student pass the exam?
  exp=Unknown/abstain got=None     rule=no_proof  prem=[]  | Q=Do all students submit all assignments?
  exp=(conflict) got=Yes      rule=PY_universal_positive  prem=[0]  | Q=Do all students improve technique?


In [61]:
# CELL 10 — Roadmap status & how this slots into the pipeline
print("""
v38 engine status:
  [DONE] FOL parse + closure (pos/neg/exist) + certificate (premises_used) + v35 convention
  [DONE] v39-lite predicate normalization (symmetric de-inflection, uniqueness gate)
  [DONE] v40 MC unique-proof solver (apply only if exactly one option proven)
  [DONE] v42 conflict policy = abstain unless a single clean projection
Integration:
  pipeline: LoRA answer -> verify_v38(question, premises_FOL[, options])
            if engine answer is not None -> override + use its premises_used (certificate)
            if None -> keep LoRA answer (engine abstains, never overwrites)
  LIVE caveat: competition input has NL premises only (no FOL). For live, either
   (a) parse NL->atoms via v39 normalization (lower recall), or (b) keep LoRA as primary
       and let v38 act only when NL maps cleanly. Do NOT assume offline recall == live recall.
Next (not yet): v43 statement-form 2.0, v44 LLM-as-parser (offline only), v45 metadata generator.
""")


v38 engine status:
  [DONE] FOL parse + closure (pos/neg/exist) + certificate (premises_used) + v35 convention
  [DONE] v39-lite predicate normalization (symmetric de-inflection, uniqueness gate)
  [DONE] v40 MC unique-proof solver (apply only if exactly one option proven)
  [DONE] v42 conflict policy = abstain unless a single clean projection
Integration:
  pipeline: LoRA answer -> verify_v38(question, premises_FOL[, options])
            if engine answer is not None -> override + use its premises_used (certificate)
            if None -> keep LoRA answer (engine abstains, never overwrites)
  LIVE caveat: competition input has NL premises only (no FOL). For live, either
   (a) parse NL->atoms via v39 normalization (lower recall), or (b) keep LoRA as primary
       and let v38 act only when NL maps cleanly. Do NOT assume offline recall == live recall.
Next (not yet): v43 statement-form 2.0, v44 LLM-as-parser (offline only), v45 metadata generator.

